# 17 · Monte Carlo Deep Dive

Use `MonteCarloPathGenerator` to simulate GBM paths, capture subsets for visualization, and compute basic path statistics.

### Learning Objectives
- Generate GBM paths with configurable drift/vol/dividend parameters
- Inspect captured paths and convert them to tabular data
- Compare distributions under different vol regimes

In [ ]:
from finstack.valuations.common.mc import MonteCarloPathGenerator
generator = MonteCarloPathGenerator()


## 1. Generate Paths
Simulate 500 GBM paths over one year, capturing 50 for inspection.

In [ ]:
paths = generator.generate_gbm_paths(
    initial_spot=100.0,
    r=0.05,
    q=0.02,
    sigma=0.25,
    time_to_maturity=1.0,
    num_steps=252,
    num_paths=500,
    capture_mode="sample",
    sample_count=50,
    seed=42,
)
print("Total paths simulated:", paths.num_paths_total)
print("Paths captured:", paths.num_captured())
print("Sampling ratio:", f"{paths.sampling_ratio():.1%}")
print("State variables:", paths.state_var_keys())
first = paths.path(0)
print("First path steps:", first.num_steps())
print("Terminal spot:", first.terminal_point().spot())


## 2. Convert to Tabular Data
`to_dict()` returns long-format data suitable for pandas/Polars. We compute simple statistics over time without additional dependencies.

In [ ]:
data = paths.to_dict()
print("Keys:", data.keys())
first_times = data["time"][:5]
first_spots = data["spot"][:5]
print("First five time points:", list(zip(first_times, first_spots)))

# Compute mean/standard deviation at a few time buckets
import math
buckets = [0.25, 0.5, 0.75, 1.0]
for target in buckets:
    values = [spot for t, spot in zip(data["time"], data["spot"]) if abs(t - target) < 1e-6]
    if values:
        mean = sum(values) / len(values)
        variance = sum((v - mean) ** 2 for v in values) / len(values)
        print(f"t={target:.2f} -> mean={mean:.2f}, std={math.sqrt(variance):.2f}")


## 3. Compare Volatility Regimes
Run two simulations with different volatilities and inspect terminal distributions.

In [ ]:
low_vol = generator.generate_gbm_paths(100.0, 0.05, 0.02, 0.15, 1.0, 252, 500, "sample", 30, 1)
high_vol = generator.generate_gbm_paths(100.0, 0.05, 0.02, 0.35, 1.0, 252, 500, "sample", 30, 2)
low_terminal = [path.terminal_point().spot() for path in low_vol.paths]
high_terminal = [path.terminal_point().spot() for path in high_vol.paths]
print("Low-vol terminal mean:", sum(low_terminal) / len(low_terminal))
print("High-vol terminal mean:", sum(high_terminal) / len(high_terminal))
print("Low-vol min/max:", min(low_terminal), max(low_terminal))
print("High-vol min/max:", min(high_terminal), max(high_terminal))


## Summary
- `MonteCarloPathGenerator` supports sampling captured paths for diagnostics while simulating large ensembles.
- `to_dict()` exposes long-format data that can be fed into pandas/Polars or analyzed manually.
- Changing volatility parameters demonstrates the widening of terminal distributions without altering drift inputs.